# AI 폰트 PoC — Korean-Diff-Font 1-shot 추론

**목표**: 사전학습된 한글 diffusion 모델로 *1 장의 참조 글리프* 에서 *9 음절* 을 같은 스타일로 생성.

**Phase 0** (이 노트북): T4 1 시간 내 추론 가능성 검증.

**성공 기준 (사용자 시각 판정)**:
1. 모델 로드 + 추론 에러 없이 완료
2. 생성된 9 글자가 *식별 가능한 한글*
3. 9 글자가 *참조와 일관된 스타일*

위 3 가지 모두 충족 → Phase 1 (전체 11,172 음절 생성) 으로.  하나라도 실패 → 종료.

*상세 근거: `docs/ai-font-poc.md`*

## 1. GPU 확인 (T4 16GB 기대)

In [ ]:
!nvidia-smi

## 2. Repo 클론

- `Korean-Diff-Font` 본체
- 사전 포함 파일: `total_kor.txt` (11,172 자), `korean_stroke.txt` (획 정의), `sample.py` (추론), `cfg/` (빈 디렉토리)

In [ ]:
import os
if not os.path.exists('Korean-Diff-Font'):
    !git clone https://github.com/ORI-Muchim/Korean-Diff-Font.git
%cd Korean-Diff-Font
!ls

## 3. 의존성 설치

`requirements.txt` 는 `pytorch==1.11.0` 으로 매우 오래된 버전을 핀.  Colab 기본 PyTorch (2.x) 와 호환 검증 필요 — *Phase 0 의 첫 위험 포인트*.

전략: 먼저 *기본 Colab PyTorch* 로 시도 → 안되면 1.11 다운그레이드.  `mpi4py` 는 OS 패키지 필요.

In [ ]:
!apt-get install -y libopenmpi-dev 2>&1 | tail -5
!pip install -q tqdm opencv-python scikit-learn pillow tensorboardX 'blobfile>=1.0.5' mpi4py attrdict pyyaml

## 3.5 attrdict Python 3.12 호환 패치

`attrdict` 라이브러리는 2019년 이후 미관리, `from collections import Mapping`
사용 — Python 3.10 부터 `collections.Mapping` 제거.  Colab Python 3.12 에서
sample.py 의 `from attrdict import AttrDict` ImportError 로 추론 자체가 깨짐.

패치: 설치된 attrdict 의 `.py` 파일에서 `from collections import (Mapping|
MutableMapping|Sequence)` 를 `from collections.abc import ...` 로 sed 치환.


In [ ]:
import glob, os, sys
patched = []
for pat in [
    '/usr/local/lib/python3.*/dist-packages/attrdict/*.py',
    '/usr/lib/python3.*/dist-packages/attrdict/*.py',
]:
    for f in glob.glob(pat):
        with open(f) as fh:
            s = fh.read()
        s2 = (s.replace('from collections import Mapping', 'from collections.abc import Mapping')
                .replace('from collections import MutableMapping', 'from collections.abc import MutableMapping')
                .replace('from collections import Sequence', 'from collections.abc import Sequence'))
        if s2 != s:
            with open(f, 'w') as fh:
                fh.write(s2)
            patched.append(f)
print(f'patched {len(patched)} file(s):')
for f in patched:
    print(f'  {f}')

if 'attrdict' in sys.modules:
    del sys.modules['attrdict']
    for m in list(sys.modules):
        if m.startswith('attrdict.'):
            del sys.modules[m]

from attrdict import AttrDict
print('\nimport verified:', AttrDict({'k': 'v'}).k)


## 4. 사전학습 체크포인트 다운로드 (498 MB)

- `ema_0.9999_800000.pt` — EMA 가중치 (추론용, 권장)
- 다른 두 파일 (`model800000.pt` 498 MB, `opt800000.pt` 921 MB) 은 *학습 재개용* 으로 추론에 불필요

In [ ]:
!mkdir -p trained_models
!wget -q --show-progress -O trained_models/ema_0.9999_800000.pt \
    https://github.com/ORI-Muchim/Korean-Diff-Font/releases/download/1.0/ema_0.9999_800000.pt
!ls -lh trained_models/

## 5. 참조 글리프 렌더링 — 1 장

1-shot 추론은 **타겟 스타일을 보여주는 글리프 1 장** 이 필요.  여기서는:
- 타겟 폰트: **나눔고딕** (Google Fonts, OFL)
- 참조 문자: **'가'** (가장 흔한 첫 글자, 모음·받침 없음)
- 크기: 128×128 (모델 학습 해상도)
- 형식: 흰 배경 + 검은 글자 (학습 데이터 형식)

*다른 폰트 / 다른 문자로 바꾸려면 셀 안의 `TARGET_FONT_URL` 과 `REF_CHAR` 수정.*

In [ ]:
import urllib.request
from PIL import Image, ImageDraw, ImageFont

TARGET_FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
REF_CHAR = '가'
IMG_SIZE = 128

urllib.request.urlretrieve(TARGET_FONT_URL, 'target_font.ttf')

def render_glyph(font_path, char, size=128):
    """한 글자를 size x size PNG (흰 배경, 검은 글자) 로 렌더.  ink-bbox 중앙 정렬."""
    render_px = size * 4
    font = ImageFont.truetype(font_path, int(render_px * 0.7))
    img = Image.new('L', (render_px, render_px), 255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), char, font=font)
    tx = (render_px - (bbox[2] - bbox[0])) / 2 - bbox[0]
    ty = (render_px - (bbox[3] - bbox[1])) / 2 - bbox[1]
    draw.text((tx, ty), char, font=font, fill=0)
    return img.resize((size, size), Image.LANCZOS)

ref_img = render_glyph('target_font.ttf', REF_CHAR, IMG_SIZE)
ref_img.save('1.png')
print(f'참조 글리프 1.png 저장 — 문자: {REF_CHAR!r} 크기: {ref_img.size}')
display(ref_img)

## 6. 생성할 문자 지정

`gen_char_kor.txt` 에 생성할 문자 나열.  Phase 0 은 9 글자 (3x3 grid):
- `가` (참조와 동일 — 모델 reconstruction 능력 확인)
- `나`, `다`, `라`, `마`, `바`, `사`, `아`, `자` (다양한 자모)

In [ ]:
GEN_CHARS = '가나다라마바사아자'
with open('gen_char_kor.txt', 'w', encoding='utf-8') as f:
    f.write(GEN_CHARS)
print(f'생성 대상 {len(GEN_CHARS)} 글자: {GEN_CHARS}')

## 6.5 디버그 — 라벨 매핑 진단

라운드 1: "가/나" 요청 → 모델이 "하/이" 출력.
**클래스 인덱스 ↔ 글자 매핑이 `total_kor.txt` 순서와 다름** 가능성.

이 셀:
- `total_kor.txt` 의 첫 30 글자
- 우리 GEN_CHARS 의 char2idx 매핑
- 라운드 1 출력 "하/이" 의 char2idx 매핑

→ 추론 *후* 결과 PNG 가 이 char2idx 와 일치하는지 시각 검증.


In [ ]:
with open('total_kor.txt', encoding='utf-8') as f:
    txt = f.read().rstrip('\n')
print(f'total_kor.txt 총 길이: {len(txt)}')
print(f'첫 30 글자: {txt[:30]!r}')
char2idx = {ch: i for i, ch in enumerate(txt)}
print('\n[요청] 가나다라마바사아자 → 클래스:')
for ch in '가나다라마바사아자':
    print(f'  {ch} → class {char2idx[ch]}')
print('\n[라운드 1 출력] 하 이 → 클래스:')
for ch in '하이':
    print(f'  {ch} → class {char2idx.get(ch, "NOT IN total_kor.txt")}')


## 7. 추론 config 작성

`cfg/test_cfg.yaml` 신규 작성 (repo 의 `cfg/` 디렉토리는 비어 있음).  파라미터는 사전학습 모델 구조에 맞춤:
- 128 채널, 128×128 이미지, DDIM 25 step
- 11,172 한글 클래스

In [ ]:
cfg = '''# Korean-Diff-Font 추론 설정 (Phase 0 디버그 라운드 2)
# batch_size=1 + num_samples=9 → 9 글자, 텐서 shape 안전
# guidance scale 3.0 → 1.5 (라운드 1 의 두꺼운 출력 완화)
model_path: './trained_models/ema_0.9999_800000.pt'
sty_img_path: './1.png'
total_txt_file: './total_kor.txt'
gen_txt_file: './gen_char_kor.txt'
stroke_path: './korean_stroke.txt'
img_save_path: './result'

image_size: 128
num_channels: 128
num_res_blocks: 3
attention_resolutions: "40,20,10"
dropout: 0.1
diffusion_steps: 1000
noise_schedule: linear
use_ddim: true
timestep_respacing: "ddim25"

classifier_free: true
cont_scale: 1.5
sk_scale: 1.5
batch_size: 1
num_samples: 9
'''
import os
os.makedirs('cfg', exist_ok=True)
with open('cfg/test_cfg.yaml', 'w') as f:
    f.write(cfg)
print('cfg/test_cfg.yaml 작성 완료')
!cat cfg/test_cfg.yaml

## 8. 추론 실행

**예상 시간**: T4 에서 글자당 5~10 초 × 9 글자 ≈ 1~2 분.

**실패 시 진단 포인트**:
- `pytorch` 버전 충돌 → 출력 traceback 확인, `pip install torch==1.11.0` 시도
- `mpi4py` import 실패 → `apt-get install libopenmpi-dev` 재실행
- OOM → `batch_size: 1` 로 낮춤
- 모델 가중치 mismatch → config 의 `num_channels` / `num_res_blocks` 확인

In [ ]:
!python sample.py --cfg_path cfg/test_cfg.yaml

## 9. 결과 시각화 — Before / After

- **Before** = 참조 글리프 1 장 (Nanum '가')
- **After** = 모델이 생성한 9 글자
- 같은 행에 *나눔고딕 원본* 도 같이 렌더 → AI 가 참조 스타일을 *얼마나 잘 흉내냈는지* 비교

In [ ]:
import os
from PIL import Image

result_files = sorted(f for f in os.listdir('result') if f.endswith('.png'))
print(f'생성된 파일 {len(result_files)} 개: {result_files[:5]}...')

GRID_COLS = 3
GRID_ROWS = 3
CELL = 128
PAD = 6

def make_strip(label, images):
    strip = Image.new('L', ((CELL + PAD) * len(images) + PAD, CELL + PAD * 2 + 20), 255)
    for i, im in enumerate(images):
        if im.size != (CELL, CELL):
            im = im.resize((CELL, CELL))
        strip.paste(im, (PAD + i * (CELL + PAD), PAD))
    return strip

nanum_images = [render_glyph('target_font.ttf', ch, CELL) for ch in GEN_CHARS]
ai_images = [Image.open(f'result/{f}') for f in result_files[:len(GEN_CHARS)]]

from PIL import ImageDraw, ImageFont as PFont
out = Image.new('L', ((CELL + PAD) * len(GEN_CHARS) + PAD, (CELL + PAD * 2) * 2 + 40), 255)
draw = ImageDraw.Draw(out)
draw.text((PAD, 4), 'Nanum (target)', fill=0)
for i, im in enumerate(nanum_images):
    out.paste(im, (PAD + i * (CELL + PAD), 24))
draw.text((PAD, CELL + PAD * 2 + 30), 'AI (generated)', fill=0)
for i, im in enumerate(ai_images):
    out.paste(im, (PAD + i * (CELL + PAD), CELL + PAD * 2 + 50))
out.save('phase0_compare.png')
display(out)

## 10. 판정 — 사용자 직접

위 비교 이미지 (`phase0_compare.png`) 를 **사람 눈으로** 보고 yes/no:

| 체크 | 통과 조건 |
|---|---|
| 모델 추론 | 9 글자 PNG 가 `result/` 에 생성됨 |
| 식별성 | 9 글자가 한글 글자로 *읽힘* (노이즈 아님) |
| 일관성 | 9 글자가 *같은 스타일* (제각각 아님) |
| 참조 유사 | AI 행이 Nanum 행과 *비슷한 느낌* |

**디버그 라운드 2 사전 정의 종결 기준** ([[feedback-foresight-before-metrics]]):

| 결과 | 다음 |
|---|---|
| 9 글자 *가나다라마바사아자* 식별 가능 + 스타일 어느 정도 따라옴 | Phase 0 통과 재판정 → Phase 1 |
| 9 글자 나오지만 *여전히 가나다 아닌 다른 글자* | **종결** — 학습 chara.txt 미공개, 라벨 매핑 복원 불가. fallback (Clova DM-Font) 의사 재확인 |
| 9 글자 *맞게* 나오지만 *스타일 제각각* | **종결** — style transfer 약함, 다른 모델 의사 재확인 |

판정 결과 채팅으로 (`phase0_compare.png` 이미지 첨부 권장).


### P1.1 11,172 음절 전체 생성

- `gen_char_kor.txt` 를 `total_kor.txt` 전체 (11,172 자) 로 교체
- `batch_size`, `num_samples` 메모리·시간 trade-off 조정
- 예상 시간: T4 1 글자 ~5 초 × 11,172 ≈ 15~20 시간 (Pro 세션 끊김 없음)

In [ ]:
# Phase 0 통과 후 주석 해제
# import shutil
# shutil.copy('total_kor.txt', 'gen_char_kor.txt')
# !python sample.py --cfg_path cfg/test_cfg.yaml

### P1.2 PNG → TTF 변환 (선택)

11,172 PNG 를 `fontTools` + `Potrace` (벡터화) 로 새 TTF 빌드.

*Phase 1.1 완료 후 별도 노트북 또는 로컬 스크립트로 처리 — 메모리 효율적.*